In [1]:
import os
import pandas as pd
import numpy as np
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from sklearn.manifold import TSNE
import plotly.express as px

In [2]:
# Cell 2: Connect and Debug
import os
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 1. Absolute Path Check (Highly Recommended)
# Replace this with the full path to your project folder if needed
db_path = os.path.abspath("f:/wellness-agent-rag/chroma_db")
print(f"Looking for database at: {db_path}")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(persist_directory=db_path, embedding_function=embeddings)

# 2. Check Count immediately
count = vectorstore._collection.count()
print(f"Documents found in collection: {count}")

if count == 0:
    print("❌ ERROR: Database is empty. Check your path!")
else:
    # 3. Fetch data with explicit include
    data = vectorstore.get(include=['embeddings', 'metadatas', 'documents'])
    print(f"Retrieved {len(data['embeddings'])} embeddings.")

Looking for database at: f:\wellness-agent-rag\chroma_db


C:\Users\panka\AppData\Local\Temp\ipykernel_10764\1385490337.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=db_path, embedding_function=embeddings)


Documents found in collection: 701
Retrieved 701 embeddings.


In [3]:
# Cell 3: Dimensionality Reduction (The "Math" bit)
# Gemini embeddings are high-dimensional. t-SNE squashes them to 3D for our eyes.
print(f"Processing {len(data['embeddings'])} vectors...")
tsne = TSNE(n_components=3, random_state=42, perplexity=30)
projections = tsne.fit_transform(np.array(data['embeddings']))

Processing 701 vectors...


In [4]:
# Cell 4: Create DataFrame for the GUI
df = pd.DataFrame(projections, columns=['x', 'y', 'z'])
df['advisor_id'] = [m.get('advisor_id') for m in data['metadatas']]
df['summary'] = [doc[:100] + "..." for doc in data['documents']]

In [7]:
# Cell 5: Interactive 3D GUI
import plotly.io as pio
fig = px.scatter_3d(
    df, x='x', y='y', z='z',
    color='x', # Colors based on position
    hover_data=['advisor_id', 'summary'],
    title="3D Advisor Vector Space"
)

fig.update_traces(marker=dict(size=5))
pio.renderers.default = "vscode"
fig.show()